In [3]:
import requests

url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
filename = 'tinyshakespeare.txt'

response = requests.get(url)
if response.status_code == 200:
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(response.text)
    print(f'{filename} downloaded successfully.')
else:
    print('Failed to download the file.')


tinyshakespeare.txt downloaded successfully.


In [8]:
# rnn_char.py
# Minimal character-level RNN on Tiny Shakespeare with train/test split

import math
import os
import torch
import torch.nn as nn

# --------------------
# Config
# --------------------
torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "cpu"

data_path = "tinyshakespeare.txt"  # ensure this file exists in the current directory
train_ratio = 0.9                  # train/test split
block_size = 128                   # sequence length
batch_size = 64
n_emb = 128
n_hidden = 256
num_layers = 1
lr = 3e-3
max_steps = 3000
eval_interval = 200
eval_batches = 10                  # how many batches to average for test loss

# --------------------
# Load and split data
# --------------------
with open(data_path, "r", encoding="utf-8") as f:
    text = f.read()

# Build vocabulary from full text (to avoid OOV in test)
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s: str) -> torch.Tensor:
    return torch.tensor([stoi[c] for c in s], dtype=torch.long)

# Train/test split on raw text
split_idx = int(len(text) * train_ratio)
train_text = text[:split_idx]
test_text  = text[split_idx:]

train_ids = encode(train_text)
test_ids  = encode(test_text)

# --------------------
# Batch sampling
# --------------------
def get_batch(data_ids: torch.Tensor, batch: int, seq_len: int):
    # Random starting positions; ensure room for seq_len+1
    idx = torch.randint(0, len(data_ids) - seq_len - 1, (batch,))
    x = torch.stack([data_ids[i : i + seq_len] for i in idx])
    y = torch.stack([data_ids[i + 1 : i + 1 + seq_len] for i in idx])
    return x.to(device), y.to(device)

# --------------------
# Model
# --------------------
class CharRNN(nn.Module):
    def __init__(self, vocab_size: int, n_emb: int, n_hidden: int, num_layers: int = 1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, n_emb)
        self.rnn = nn.RNN(n_emb, n_hidden, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(n_hidden, vocab_size)

    def forward(self, x, h=None):
        x = self.embed(x)                  # (B, T, n_emb)
        out, h = self.rnn(x, h)            # (B, T, n_hidden)
        logits = self.fc(out)              # (B, T, vocab_size)
        return logits, h

model = CharRNN(vocab_size, n_emb, n_hidden, num_layers).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

# --------------------
# Evaluation
# --------------------
@torch.no_grad()
def evaluate_test_loss():
    model.eval()
    losses = []
    for _ in range(eval_batches):
        x, y = get_batch(test_ids, batch_size, block_size)
        logits, _ = model(x)
        loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))
        losses.append(loss.item())
    avg = sum(losses) / len(losses)
    ppl = math.exp(avg)
    return avg, ppl

# --------------------
# Training loop
# --------------------
for step in range(1, max_steps + 1):
    model.train()
    x, y = get_batch(train_ids, batch_size, block_size)
    optimizer.zero_grad()
    logits, _ = model(x)
    loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))
    loss.backward()
    optimizer.step()

    if step % eval_interval == 0 or step == 1:
        test_loss, test_ppl = evaluate_test_loss()
        print(f"step {step:5d} | train loss {loss.item():.4f} | test loss {test_loss:.4f} | test ppl {test_ppl:.2f}")

# Final evaluation
final_loss, final_ppl = evaluate_test_loss()
print(f"Final test loss: {final_loss:.4f} | Final test perplexity: {final_ppl:.2f}")


step     1 | train loss 4.1810 | test loss 3.8166 | test ppl 45.45
step   200 | train loss 1.7005 | test loss 1.8319 | test ppl 6.25
step   200 | train loss 1.7005 | test loss 1.8319 | test ppl 6.25
step   400 | train loss 1.5873 | test loss 1.7168 | test ppl 5.57
step   400 | train loss 1.5873 | test loss 1.7168 | test ppl 5.57
step   600 | train loss 1.4892 | test loss 1.6706 | test ppl 5.32
step   600 | train loss 1.4892 | test loss 1.6706 | test ppl 5.32
step   800 | train loss 1.4565 | test loss 1.6632 | test ppl 5.28
step   800 | train loss 1.4565 | test loss 1.6632 | test ppl 5.28
step  1000 | train loss 1.4951 | test loss 1.6398 | test ppl 5.15
step  1000 | train loss 1.4951 | test loss 1.6398 | test ppl 5.15
step  1200 | train loss 1.4302 | test loss 1.6298 | test ppl 5.10
step  1200 | train loss 1.4302 | test loss 1.6298 | test ppl 5.10
step  1400 | train loss 1.3961 | test loss 1.6346 | test ppl 5.13
step  1400 | train loss 1.3961 | test loss 1.6346 | test ppl 5.13
step  160

In [9]:
# Save the trained model weights
model_save_path = "char_rnn_model.pth"

# Save only the model state dictionary (recommended approach)
torch.save(model.state_dict(), model_save_path)
print(f"Model weights saved to {model_save_path}")

# Optional: Save additional training information
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'vocab_size': vocab_size,
    'n_emb': n_emb,
    'n_hidden': n_hidden,
    'num_layers': num_layers,
    'stoi': stoi,
    'itos': itos,
    'train_loss': loss.item(),
    'test_loss': test_loss,
    'test_ppl': test_ppl
}
torch.save(checkpoint, "char_rnn_complete.pth")
print(f"Complete checkpoint saved to char_rnn_complete.pth")

Model weights saved to char_rnn_model.pth
Complete checkpoint saved to char_rnn_complete.pth


In [10]:
# Example: How to load the saved model weights later

# Recreate the model architecture (must match the saved model)
model = CharRNN(vocab_size, n_emb, n_hidden, num_layers).to(device)

# Load the saved weights
model.load_state_dict(torch.load("char_rnn_model.pth", map_location=device))
model.eval()  # Set to evaluation mode

print("Model weights loaded successfully!")
print(f"Model is on device: {next(model.parameters()).device}")

Model weights loaded successfully!
Model is on device: cpu


In [11]:
final_loss, final_ppl = evaluate_test_loss()
print(f"Final test loss: {final_loss:.4f} | Final test perplexity: {final_ppl:.2f}")

Final test loss: 1.5938 | Final test perplexity: 4.92


In [23]:
encode("\n")

tensor([0])

In [ ]:

import math
import torch
import torch.nn as nn

# --------------------
# Config
# --------------------
torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "cpu"

data_path = "tinyshakespeare.txt"  # file must exist in current directory
train_ratio = 0.9
bptt_len = 128        # window length for normal backprop; increase for longer credit assignment
n_emb = 128
n_hidden = 256
num_layers = 1
epochs = 1            # increase for more passes

model = CharRNN(vocab_size, n_emb, n_hidden, num_layers).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

model.load_state_dict(torch.load("char_rnn_model.pth", map_location=device))

# --------------------
# Training: single continuous stream with truncated BPTT
# --------------------
for epoch in range(epochs):
    model.train()
    h = torch.zeros(num_layers, 1, n_hidden, device=device)
    total_loss = 0.0
    total_tokens = 0

    i = 0
    while i + 1 < len(train_ids):
        # Prepare a window [i, i+bptt_len)
        j = min(i + bptt_len, len(train_ids) - 1)
        x_seq = train_ids[i:j].to(device)           # (T,)
        y_seq = train_ids[i+1:j+1].to(device)       # (T,)

        if x_seq.numel() == 0:
            break

        x_seq = x_seq.unsqueeze(1)  # (T, 1)
        y_seq = y_seq.unsqueeze(1)  # (T, 1)

        optimizer.zero_grad()
        logits, h = model(x_seq, h)  # logits: (T, 1, V)
        loss = criterion(logits.reshape(-1, vocab_size), y_seq.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        # Detach hidden to cut the graph, carry state forward
        h = h.detach()

        total_loss += loss.item() * (j - i)
        total_tokens += (j - i)
        i = j

    avg = total_loss / max(1, total_tokens)
    print(f"epoch {epoch+1} | avg train loss {avg:.4f} | ppl {math.exp(avg):.2f}")

# --------------------
# Evaluation: sequential stream, no grad
# --------------------
@torch.no_grad()
def evaluate(ids):
    model.eval()
    h = torch.zeros(num_layers, 1, n_hidden, device=device)
    total = 0.0
    tokens = 0
    i = 0
    while i + 1 < len(ids):
        j = min(i + bptt_len, len(ids) - 1)
        x_seq = ids[i:j].to(device).unsqueeze(1)
        y_seq = ids[i+1:j+1].to(device).unsqueeze(1)
        if x_seq.numel() == 0:
            break
        logits, h = model(x_seq, h)
        loss = criterion(logits.reshape(-1, vocab_size), y_seq.reshape(-1))
        total += loss.item() * (j - i)
        tokens += (j - i)
        i = j
    avg = total / max(1, tokens)
    return avg, math.exp(avg)

test_loss, test_ppl = evaluate(test_ids)
print(f"Test loss: {test_loss:.4f} | Test perplexity: {test_ppl:.2f}")
